# CASMI 2026 inference (b1_library)

Generated by `scripts/package_kaggle.py` from git commit `8c486ac`. Offline: RDKit from `metric/rdkit-2026-3-3-wheel`, code from `ograhulrajkumar/casmi-code`, artefacts from the datasets in `ARTEFACTS`. Writes `/kaggle/working/submission.csv`. Edit only the settings cell.

In [ ]:
# ---- the only settings ---------------------------------------------------------------
from pathlib import Path
CONFIG = 'b1_library'                     # configs/<CONFIG>.yaml in the code dataset
OWNER = 'ograhulrajkumar'
COMPETITION = 'enveda-CASMI26-molecule-id-mass-spectra'
CODE_DATASET = 'casmi-code'
ARTEFACTS = {
    "processed/db/np": "casmi-db-np",
    "processed/db/pubchem": "casmi-db-pubchem",
    "processed/library": "casmi-library"
}  # subdir under CASMI_ARTEFACTS -> dataset
RDKIT_WHEEL_DATASET = 'metric/rdkit-2026-3-3-wheel'
WHEEL_DATASETS = []             # extra wheel datasets, installed with --no-index
WORK = Path("/kaggle/working")
RUN = WORK / "casmi_run"               # CASMI_ROOT: logs and scratch
SUBMISSION = WORK / "submission.csv"
FALLBACK_SMILES = "Oc1ccccc1"          # only if the whole pipeline cannot run

In [ ]:
import os, sys, time, json, glob, shutil, hashlib, subprocess, traceback
from pathlib import Path

INPUT = Path("/kaggle/input")


def find_dataset(slug, owner=None):
    """Mount directory of an attached dataset, found by slug (any Kaggle mount layout)."""
    for cand in ([INPUT / "datasets" / owner / slug] if owner else []) + [INPUT / slug]:
        if cand.is_dir():
            return cand
    hits = []
    for dirpath, dirnames, _ in os.walk(INPUT):
        depth = len(Path(dirpath).relative_to(INPUT).parts)
        if depth >= 4:
            dirnames[:] = []
            continue
        for d in sorted(dirnames):
            if d.lower() == slug.lower():
                hits.append(Path(dirpath) / d)
    if owner:
        hits.sort(key=lambda p: (owner not in p.parts, len(p.parts), str(p)))
    return hits[0] if hits else None


def find_test(competition):
    """The competition's test.parquet (replaced by the hidden test on a scoring rerun)."""
    comp = find_dataset(competition)
    if comp is not None:
        hits = sorted(comp.rglob("test.parquet"))
        if hits:
            return hits[0]
    hits = sorted(INPUT.rglob("test.parquet"))
    if not hits:
        raise FileNotFoundError("no test.parquet under /kaggle/input")
    return hits[0]

In [ ]:
T0 = time.time()
print("python", sys.version.split()[0], "| cpus", os.cpu_count())
TEST = find_test(COMPETITION)  # without it no submission is possible at all
print("test:", TEST)
CODE = None
SETUP_ERROR = None


def pip_install(wheel_dir, *reqs):
    cmd = [sys.executable, "-m", "pip", "install", "--no-deps", "--no-index", "--quiet",
           f"--find-links={wheel_dir}", *reqs]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(" ".join(cmd[3:]), "->", r.returncode, r.stderr[-800:])
    return r.returncode == 0


def setup():
    global CODE
    # RDKit from the metric's wheel dataset (internet is off).
    try:
        import rdkit  # noqa: F401
    except ImportError:
        owner, slug = RDKIT_WHEEL_DATASET.split("/")
        wd = find_dataset(slug, owner)
        print("rdkit wheel dir:", wd)
        pip_install(wd, "rdkit==2026.3.3")
    for ref in WHEEL_DATASETS:
        owner, slug = ref.split("/")
        wd = find_dataset(slug, owner)
        wheels = sorted(str(p) for p in wd.rglob("*.whl")) if wd else []
        if wheels:
            pip_install(wd, *wheels)
    import importlib
    importlib.invalidate_caches()
    import rdkit
    print("rdkit", rdkit.__version__)

    # Code dataset on sys.path (and PYTHONPATH for worker processes).
    CODE = find_dataset(CODE_DATASET, OWNER)
    if CODE is None or not (CODE / "casmi" / "__init__.py").is_file():
        raise FileNotFoundError(f"code dataset {CODE_DATASET} not found ({CODE})")
    sys.path.insert(0, str(CODE))
    os.environ["PYTHONPATH"] = str(CODE) + os.pathsep + os.environ.get("PYTHONPATH", "")

    # Artefact tree: symlinks to the mounted datasets, plus writable dirs (mkey cache).
    art = RUN / "artefacts"
    if art.exists():
        shutil.rmtree(art)
    for sub, slug in sorted(ARTEFACTS.items()):
        src = find_dataset(slug, OWNER)
        if src is None:
            print(f"WARNING: dataset {slug} not attached; {sub} missing")
            continue
        dst = art / sub
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.symlink_to(src, target_is_directory=True)
        print(f"{sub:24s} -> {src}")
    (art / "processed" / "cache").mkdir(parents=True, exist_ok=True)
    (art / "models").mkdir(parents=True, exist_ok=True)
    os.environ.update({
        "CASMI_ROOT": str(RUN),
        "CASMI_ARTEFACTS": str(art),
        "CASMI_RAW": str(TEST.parent),
        "CASMI_CONFIGS": str(CODE / "configs"),
        "NUMBA_CACHE_DIR": str(WORK / "numba_cache"),
    })
    Path(os.environ["NUMBA_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)


try:
    setup()
except Exception:
    SETUP_ERROR = traceback.format_exc()
    print(SETUP_ERROR)

In [ ]:
def write_fallback(reason):
    """A valid file no matter what: one guess per test molecule."""
    import pandas as pd
    ids = sorted(set(pd.read_parquet(TEST, columns=["molecule_id"])["molecule_id"].astype(str)))
    pd.DataFrame({"molecule_id": ids, "smiles": [FALLBACK_SMILES] * len(ids)}).to_csv(
        SUBMISSION, index=False, lineterminator="\n")
    print(f"FALLBACK submission written ({reason}): {len(ids)} molecules")


test_df = None
cfg = None
try:
    if SETUP_ERROR:
        raise RuntimeError("setup failed")
    from casmi import io, log, paths, pipeline
    from casmi.config import load_config
    lg = log.get_logger(f"kaggle_{CONFIG}")
    print("paths:", json.dumps(paths.as_dict(), indent=1))
    cfg = load_config(CODE / "configs" / f"{CONFIG}.yaml")
    print("config overrides:", cfg.overrides())
    pipeline.limit_threads(cfg.n_jobs)
    test_df = io.read_test(TEST)
    res = pipeline.run(test_df, cfg)
    io.write_submission(res.preds, SUBMISSION, test_df=test_df, max_guesses=cfg.top_k)
    rep = RUN / "logs" / f"kaggle_{CONFIG}_molecules.csv"
    rep.parent.mkdir(parents=True, exist_ok=True)
    res.molecules.to_csv(rep, index=False, lineterminator="\n")
    print(f"pipeline done in {res.wall_s:.1f} s; peak RSS {log.peak_rss_mb():.0f} MB")
except Exception:
    traceback.print_exc()
    write_fallback("pipeline failed")

In [ ]:
try:
    from casmi import io
    report = io.validate_submission(SUBMISSION, test_df=test_df,
                                    max_guesses=cfg.top_k if cfg is not None else 25)
    print("validate_submission:", report)
except Exception:
    traceback.print_exc()
    write_fallback("validation failed")
    import pandas as pd
    chk = pd.read_csv(SUBMISSION)
    assert chk["molecule_id"].is_unique and chk["smiles"].notna().all()

shutil.rmtree(RUN / "artefacts", ignore_errors=True)  # symlinks into /kaggle/input
data = SUBMISSION.read_bytes()
print("submission sha256", hashlib.sha256(data).hexdigest(), f"{len(data)} bytes")
print(data.decode()[:600])
print(f"total runtime {time.time() - T0:.1f} s")